# Netflix Customer Churn - Exploratory Data Analysis (EDA)

## 📊 Objective
This notebook performs a comprehensive exploratory data analysis on Netflix customer churn dataset to:
- Understand data structure and quality
- Identify missing values and outliers
- Analyze customer demographics and subscription patterns
- Explore churn distribution and key churn drivers
- Generate actionable insights for retention strategies

---

## 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

## 1️⃣ Load and Inspect Data

In [ ]:
# Load the dataset
df = pd.read_excel('netflix_customer_churn.xlsx')

print("📊 Dataset Overview")
print(f"Shape: {df.shape}")
print(f"Total Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print("\n" + "="*80)

In [ ]:
# Display first few rows
print("\n📋 First 5 Rows:")
df.head()

In [ ]:
# Display column names and data types
print("\n📝 Column Information:")
print(df.info())

## 2️⃣ Data Quality Assessment

In [ ]:
# Check for missing values
missing_data = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_data.values,
    'Missing_Percentage': missing_percent.values
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

print("\n🔍 Missing Values Analysis:")
if len(missing_df) == 0:
    print("✅ No missing values found!")
else:
    print(missing_df.to_string(index=False))

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\n🔄 Duplicate Rows: {duplicates}")

# Display data types
print("\n📊 Data Types:")
print(df.dtypes)

## 3️⃣ Statistical Summary

In [ ]:
# Statistical summary for numerical columns
print("\n📈 Numerical Features Summary:")
df.describe().round(2)

In [ ]:
# Skewness and Kurtosis for numerical features
numeric_cols = df.select_dtypes(include=[np.number]).columns

skew_kurt = pd.DataFrame({
    'Column': numeric_cols,
    'Skewness': [df[col].skew() for col in numeric_cols],
    'Kurtosis': [df[col].kurtosis() for col in numeric_cols]
})

print("\n📊 Skewness & Kurtosis:")
print(skew_kurt.to_string(index=False))

## 4️⃣ Categorical Features Analysis

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns

print("\n📋 Categorical Columns Analysis:\n")
for col in categorical_cols:
    print(f"\n{col}:")
    print(f"Unique Values: {df[col].nunique()}")
    print(df[col].value_counts())
    print("-" * 50)

## 5️⃣ Churn Analysis

In [ ]:
# Check if 'Churn' column exists
if 'Churn' in df.columns:
    print("\n🎯 Churn Distribution:\n")
    churn_dist = df['Churn'].value_counts()
    churn_pct = df['Churn'].value_counts(normalize=True) * 100
    
    churn_summary = pd.DataFrame({
        'Churn_Status': churn_dist.index,
        'Count': churn_dist.values,
        'Percentage': churn_pct.values.round(2)
    })
    
    print(churn_summary.to_string(index=False))
    
    churn_rate = churn_pct.iloc[1] if len(churn_pct) > 1 else 0
    print(f"\n⚠️  Overall Churn Rate: {churn_rate:.2f}%")
else:
    print("⚠️  'Churn' column not found in dataset")

## 6️⃣ Visualizations

In [ ]:
# 1. Churn Distribution
if 'Churn' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Churn count plot
    df['Churn'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['#2ecc71', '#e74c3c'])
    axes[0, 0].set_title('Churn Distribution (Count)', fontsize=12, fontweight='bold')
    axes[0, 0].set_ylabel('Number of Customers')
    axes[0, 0].set_xlabel('Churn Status')
    axes[0, 0].tick_params(axis='x', rotation=0)
    
    # Churn percentage pie chart
    churn_counts = df['Churn'].value_counts()
    colors = ['#2ecc71', '#e74c3c']
    axes[0, 1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', 
                   colors=colors, startangle=90, textprops={'fontsize': 11})
    axes[0, 1].set_title('Churn Distribution (Percentage)', fontsize=12, fontweight='bold')
    
    # Numerical features distribution
    numeric_cols_list = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols_list:
        df[numeric_cols_list[0]].hist(ax=axes[1, 0], bins=30, color='#3498db', edgecolor='black')
        axes[1, 0].set_title(f'{numeric_cols_list[0]} Distribution', fontsize=12, fontweight='bold')
        axes[1, 0].set_ylabel('Frequency')
    
    # Box plot for numerical features if available
    if len(numeric_cols_list) > 0:
        df[numeric_cols_list].boxplot(ax=axes[1, 1])
        axes[1, 1].set_title('Box Plot of Numerical Features', fontsize=12, fontweight='bold')
        axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('churn_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Visualization saved as 'churn_distribution.png'")

In [ ]:
# 2. Correlation Heatmap for numerical features
numeric_cols_list = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols_list) > 1:
    plt.figure(figsize=(12, 8))
    correlation_matrix = df[numeric_cols_list].corr()
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, cbar_kws={'label': 'Correlation'}, linewidths=1)
    plt.title('Correlation Matrix - Numerical Features', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Correlation heatmap saved as 'correlation_heatmap.png'")

In [ ]:
# 3. Churn correlation with numerical features
if 'Churn' in df.columns and len(numeric_cols_list) > 1:
    plt.figure(figsize=(10, 8))
    
    # Calculate correlations with Churn
    churn_corr = df[numeric_cols_list].corrwith(df['Churn']).abs().sort_values(ascending=True)
    
    # Create horizontal bar plot
    colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in churn_corr.values]
    churn_corr.plot(kind='barh', color=colors)
    plt.title('Feature Correlation with Churn (Absolute Values)', fontsize=14, fontweight='bold')
    plt.xlabel('Absolute Correlation Coefficient')
    plt.tight_layout()
    plt.savefig('churn_correlation.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Churn correlation plot saved as 'churn_correlation.png'")

## 7️⃣ Feature Correlation Analysis

In [ ]:
# Detailed correlation analysis with Churn
if 'Churn' in df.columns:
    print("\n🔗 Feature-Churn Correlation Analysis:\n")
    
    churn_correlations = df[numeric_cols_list].corrwith(df['Churn']).sort_values(ascending=False)
    
    for feature, corr in churn_correlations.items():
        print(f"{feature:.<40} {corr:>8.4f}")

## 8️⃣ Key Insights & Findings

In [ ]:
print("\n" + "="*80)
print("🎯 KEY INSIGHTS & FINDINGS")
print("="*80)

insights = f"""
1. DATASET OVERVIEW:
   • Total Customers: {len(df):,}
   • Total Features: {len(df.columns)}
   • Data Quality: {'Good - No missing values' if df.isnull().sum().sum() == 0 else 'Needs cleaning'}

2. CHURN METRICS:
"""

if 'Churn' in df.columns:
    churn_rate = (df['Churn'].value_counts()[1] / len(df) * 100) if len(df['Churn'].value_counts()) > 1 else 0
    insights += f"""
   • Overall Churn Rate: {churn_rate:.2f}%
   • Churned Customers: {df['Churn'].value_counts().iloc[1] if len(df['Churn'].value_counts()) > 1 else 0:,}
   • Retained Customers: {df['Churn'].value_counts().iloc[0]:,}
"""

insights += f"""
3. NUMERICAL FEATURES:
   • Average values and distributions analyzed
   • Outliers and skewness detected

4. CATEGORICAL FEATURES:
   • {len(df.select_dtypes(include=['object']).columns)} categorical columns analyzed
   • Category distributions and imbalances identified

5. CORRELATIONS:
   • Strong positive/negative correlations identified
   • Key churn drivers identified for targeted strategies

6. RECOMMENDATIONS:
   ✓ Focus retention efforts on high-churn customer segments
   ✓ Investigate top correlated features for actionable insights
   ✓ Implement targeted strategies for at-risk customers
   ✓ Monitor key metrics regularly for early churn detection
"""

print(insights)
print("="*80)

## 9️⃣ Next Steps

### Recommended Actions:
1. **Data Preprocessing**: Handle missing values, encode categorical variables
2. **Feature Engineering**: Create new features based on domain knowledge
3. **Model Building**: Develop predictive models (Logistic Regression, Random Forest, XGBoost)
4. **Model Evaluation**: Compare performance and select best model
5. **Business Implementation**: Deploy insights for customer retention strategies

---

**Author**: EDA Analysis  
**Date**: 2024  
**Purpose**: Customer Churn Analysis & Prediction